# 04: Monte Carlo Simulation Engine

Complete guide to the MonteCarloEngine - the heart of Argo.

## What You'll Learn

- **MonteCarloEngine:** Core simulation framework
- **Input Variables:** Uncertain inputs with probability distributions
- **Formula Variables:** Calculated outputs with dependencies
- **Correlations:** Modeling dependencies between variables
- **Real Applications:** Revenue models, project schedules, portfolio analysis

## Prerequisites

- Argo package built (`npm run build --workspace=@argo/core`)
- TypeScript kernel (tslab) installed
- Familiarity with distributions (Notebook 01) and statistics (Notebook 02-03)

## Setup and Imports

In [ ]:
// Import Argo simulation engine
const argo = require('../packages/argo-core/dist/index');

const {
  // Simulation Engine
  MonteCarloEngine,
  
  // Distributions
  NormalDistribution,
  UniformDistribution,
  TriangularDistribution,
  PERTDistribution,
  
  // Utilities
  SimpleRNG,
  
  // Statistics
  mean,
  median,
  standardDeviation,
  percentile,
  valueAtRisk,
  conditionalVaR
} = argo;

console.log('✅ Monte Carlo simulation engine loaded!');
console.log('Ready to run simulations!');

---

# Part 1: Simple Simulation

Start with a basic simulation: single input variable, single output.

In [ ]:
console.log('=== Simple Simulation: Single Variable ===\n');

// Create simulation engine with seeded RNG
const engine1 = new MonteCarloEngine(new SimpleRNG(42));

// Define simulation: Revenue with normal distribution
const result1 = engine1.simulate({
  iterations: 10000,
  variables: [
    {
      name: 'Revenue',
      type: 'input',
      distribution: new NormalDistribution(1000000, 150000)
    }
  ]
});

// Analyze results
const revenueSamples = result1.samples['Revenue'];
console.log('Simulation Results:');
console.log(`  Iterations: ${result1.iterationsCompleted.toLocaleString()}`);
console.log(`  Execution time: ${result1.executionTimeMs}ms`);
console.log(`  Performance: ${Math.round(result1.iterationsCompleted / (result1.executionTimeMs / 1000)).toLocaleString()} iterations/second\n`);

console.log('Revenue Statistics:');
console.log(`  Mean: $${(mean(revenueSamples) / 1000).toFixed(0)}k`);
console.log(`  Median: $${(median(revenueSamples) / 1000).toFixed(0)}k`);
console.log(`  Std Dev: $${(standardDeviation(revenueSamples) / 1000).toFixed(0)}k`);
console.log(`  5th percentile: $${(percentile(revenueSamples, 5) / 1000).toFixed(0)}k`);
console.log(`  95th percentile: $${(percentile(revenueSamples, 95) / 1000).toFixed(0)}k`);

---

# Part 2: Multiple Variables with Formulas

Model relationships: Revenue - Cost = Profit

In [ ]:
console.log('\n=== Multiple Variables: Revenue - Cost = Profit ===\n');

const engine2 = new MonteCarloEngine(new SimpleRNG(123));

const result2 = engine2.simulate({
  iterations: 10000,
  variables: [
    // Input variables (uncertainties)
    {
      name: 'Revenue',
      type: 'input',
      distribution: new NormalDistribution(1000000, 150000)
    },
    {
      name: 'Cost',
      type: 'input',
      distribution: new NormalDistribution(700000, 100000)
    },
    // Formula variable (calculated)
    {
      name: 'Profit',
      type: 'formula',
      formula: 'Revenue - Cost'
    }
  ]
});

// Analyze profit
const profitSamples = result2.samples['Profit'];
console.log('Profit Analysis:');
console.log(`  Mean: $${(mean(profitSamples) / 1000).toFixed(0)}k`);
console.log(`  Median: $${(median(profitSamples) / 1000).toFixed(0)}k`);
console.log(`  Std Dev: $${(standardDeviation(profitSamples) / 1000).toFixed(0)}k\n`);

// Risk metrics
const profitVaR95 = valueAtRisk(profitSamples, 0.95);
const profitCVaR95 = conditionalVaR(profitSamples, 0.95);
console.log('Profit Risk Metrics:');
console.log(`  VaR (95%): $${(profitVaR95 / 1000).toFixed(0)}k`);
console.log(`  CVaR (95%): $${(profitCVaR95 / 1000).toFixed(0)}k\n`);

// Probability of loss
const losses = profitSamples.filter(p => p < 0).length;
console.log(`Probability of Loss: ${(losses / profitSamples.length * 100).toFixed(1)}%`);

---

# Part 3: Complex Dependencies

Project cost model with multiple dependencies.

In [ ]:
console.log('\n=== Complex Dependencies: Project Cost Model ===\n');

const engine3 = new MonteCarloEngine(new SimpleRNG(456));

const result3 = engine3.simulate({
  iterations: 10000,
  variables: [
    // Independent inputs
    {
      name: 'BaseCost',
      type: 'input',
      distribution: new TriangularDistribution(400000, 500000, 700000)
    },
    {
      name: 'TeamSize',
      type: 'input',
      distribution: new PERTDistribution(100, 120, 180)
    },
    {
      name: 'HourlyRate',
      type: 'input',
      distribution: new NormalDistribution(150, 20)
    },
    {
      name: 'Hours',
      type: 'input',
      distribution: new TriangularDistribution(1500, 2000, 3000)
    },
    
    // Calculated values (dependency chain)
    {
      name: 'LaborCost',
      type: 'formula',
      formula: 'TeamSize * HourlyRate * Hours'
    },
    {
      name: 'TotalCost',
      type: 'formula',
      formula: 'BaseCost + LaborCost'
    },
    {
      name: 'CostPerPerson',
      type: 'formula',
      formula: 'TotalCost / TeamSize'
    }
  ]
});

console.log('Execution Details:');
console.log(`  Variables: ${Object.keys(result3.samples).length}`);
console.log(`  Iterations: ${result3.iterationsCompleted.toLocaleString()}`);
console.log(`  Time: ${result3.executionTimeMs}ms\n`);

// Analyze each component
const totalCostSamples = result3.samples['TotalCost'];
const laborCostSamples = result3.samples['LaborCost'];
const baseCostSamples = result3.samples['BaseCost'];

console.log('Cost Breakdown:');
console.log(`  Base Cost:  $${(mean(baseCostSamples) / 1000).toFixed(0)}k ± $${(standardDeviation(baseCostSamples) / 1000).toFixed(0)}k`);
console.log(`  Labor Cost: $${(mean(laborCostSamples) / 1000).toFixed(0)}k ± $${(standardDeviation(laborCostSamples) / 1000).toFixed(0)}k`);
console.log(`  Total Cost: $${(mean(totalCostSamples) / 1000).toFixed(0)}k ± $${(standardDeviation(totalCostSamples) / 1000).toFixed(0)}k\n`);

console.log('Total Cost Percentiles:');
console.log(`  10th: $${(percentile(totalCostSamples, 10) / 1000).toFixed(0)}k`);
console.log(`  50th: $${(percentile(totalCostSamples, 50) / 1000).toFixed(0)}k`);
console.log(`  90th: $${(percentile(totalCostSamples, 90) / 1000).toFixed(0)}k`);

---

# Part 4: Correlated Variables

Model dependencies using correlation matrices.

In [ ]:
console.log('\n=== Correlated Variables: Market Risk ===\n');

const engine4 = new MonteCarloEngine(new SimpleRNG(789));

// Simulate two correlated market factors
const result4 = engine4.simulate({
  iterations: 10000,
  variables: [
    {
      name: 'StockA',
      type: 'input',
      distribution: new NormalDistribution(0.10, 0.20) // 10% return, 20% vol
    },
    {
      name: 'StockB',
      type: 'input',
      distribution: new NormalDistribution(0.08, 0.15) // 8% return, 15% vol
    },
    {
      name: 'Portfolio',
      type: 'formula',
      formula: '0.6 * StockA + 0.4 * StockB' // 60/40 portfolio
    }
  ],
  correlations: [
    { var1: 'StockA', var2: 'StockB', correlation: 0.7 } // Positive correlation
  ]
});

const stockASamples = result4.samples['StockA'];
const stockBSamples = result4.samples['StockB'];
const portfolioSamples = result4.samples['Portfolio'];

console.log('Individual Assets:');
console.log(`  Stock A: ${(mean(stockASamples)*100).toFixed(2)}% ± ${(standardDeviation(stockASamples)*100).toFixed(2)}%`);
console.log(`  Stock B: ${(mean(stockBSamples)*100).toFixed(2)}% ± ${(standardDeviation(stockBSamples)*100).toFixed(2)}%\n`);

console.log('Portfolio (60% A, 40% B):');
console.log(`  Return: ${(mean(portfolioSamples)*100).toFixed(2)}%`);
console.log(`  Volatility: ${(standardDeviation(portfolioSamples)*100).toFixed(2)}%\n`);

// Verify correlation
let sumXY = 0, sumX = 0, sumY = 0, sumX2 = 0, sumY2 = 0;
const n = stockASamples.length;
for (let i = 0; i < n; i++) {
  sumXY += stockASamples[i] * stockBSamples[i];
  sumX += stockASamples[i];
  sumY += stockBSamples[i];
  sumX2 += stockASamples[i] * stockASamples[i];
  sumY2 += stockBSamples[i] * stockBSamples[i];
}
const correlation = (n * sumXY - sumX * sumY) / Math.sqrt((n * sumX2 - sumX * sumX) * (n * sumY2 - sumY * sumY));

console.log('Correlation:');
console.log(`  Target: 0.70`);
console.log(`  Actual: ${correlation.toFixed(3)}`);
console.log(`  ✓ Correlation preserved in simulation`);

---

# Part 5: Progress Reporting

Monitor long-running simulations.

In [ ]:
console.log('\n=== Progress Reporting: Large Simulation ===\n');

const engine5 = new MonteCarloEngine(new SimpleRNG(999));

let lastProgress = 0;
const result5 = engine5.simulate(
  {
    iterations: 50000,
    variables: [
      {
        name: 'Revenue',
        type: 'input',
        distribution: new NormalDistribution(1000000, 200000)
      },
      {
        name: 'Cost',
        type: 'input',
        distribution: new TriangularDistribution(500000, 700000, 900000)
      },
      {
        name: 'Profit',
        type: 'formula',
        formula: 'Revenue - Cost'
      }
    ]
  },
  // Progress callback
  (completed, total) => {
    const progress = Math.floor((completed / total) * 100);
    if (progress >= lastProgress + 20) {
      console.log(`  Progress: ${progress}% (${completed.toLocaleString()}/${total.toLocaleString()})`);
      lastProgress = progress;
    }
  }
);

console.log('\nSimulation Complete!');
console.log(`  Total iterations: ${result5.iterationsCompleted.toLocaleString()}`);
console.log(`  Execution time: ${result5.executionTimeMs}ms`);
console.log(`  Performance: ${Math.round(result5.iterationsCompleted / (result5.executionTimeMs / 1000)).toLocaleString()} iterations/second`);

---

# Part 6: Real-World Example

## Software Project Risk Simulation

In [ ]:
console.log('\n========================================');
console.log('  SOFTWARE PROJECT RISK SIMULATION');
console.log('========================================\n');

const projectEngine = new MonteCarloEngine(new SimpleRNG(2024));

const projectResult = projectEngine.simulate({
  iterations: 20000,
  variables: [
    // Development estimates (story points → weeks)
    {
      name: 'Frontend',
      type: 'input',
      distribution: new PERTDistribution(8, 12, 20) // weeks
    },
    {
      name: 'Backend',
      type: 'input',
      distribution: new PERTDistribution(10, 15, 25)
    },
    {
      name: 'Testing',
      type: 'input',
      distribution: new TriangularDistribution(4, 6, 10)
    },
    
    // Total duration (critical path)
    {
      name: 'Duration',
      type: 'formula',
      formula: 'Frontend + Backend + Testing'
    },
    
    // Cost calculations
    {
      name: 'TeamSize',
      type: 'input',
      distribution: new NormalDistribution(5, 1)
    },
    {
      name: 'WeeklyCost',
      type: 'input',
      distribution: new NormalDistribution(25000, 3000)
    },
    {
      name: 'TotalCost',
      type: 'formula',
      formula: 'Duration * WeeklyCost'
    }
  ],
  correlations: [
    { var1: 'Frontend', var2: 'Backend', correlation: 0.6 }, // Similar complexity
    { var1: 'Backend', var2: 'Testing', correlation: 0.5 }  // More backend = more testing
  ]
});

const duration = projectResult.samples['Duration'];
const cost = projectResult.samples['TotalCost'];

console.log('1. Timeline Analysis:');
console.log(`   Expected duration: ${mean(duration).toFixed(1)} weeks`);
console.log(`   Median: ${median(duration).toFixed(1)} weeks`);
console.log(`   Std deviation: ${standardDeviation(duration).toFixed(1)} weeks\n`);

console.log('2. Timeline Percentiles:');
console.log(`   50% confidence: ${percentile(duration, 50).toFixed(1)} weeks`);
console.log(`   80% confidence: ${percentile(duration, 80).toFixed(1)} weeks`);
console.log(`   95% confidence: ${percentile(duration, 95).toFixed(1)} weeks\n`);

console.log('3. Cost Analysis:');
console.log(`   Expected cost: $${(mean(cost) / 1000).toFixed(0)}k`);
console.log(`   Median cost: $${(median(cost) / 1000).toFixed(0)}k`);
console.log(`   VaR (95%): $${(valueAtRisk(cost, 0.95) / 1000).toFixed(0)}k`);
console.log(`   CVaR (95%): $${(conditionalVaR(cost, 0.95) / 1000).toFixed(0)}k\n`);

console.log('4. Budget Scenarios:');
const budget800k = cost.filter(c => c <= 800000).length / cost.length;
const budget1M = cost.filter(c => c <= 1000000).length / cost.length;
const budget1_2M = cost.filter(c => c <= 1200000).length / cost.length;
console.log(`   P(Cost ≤ $800k):  ${(budget800k * 100).toFixed(1)}%`);
console.log(`   P(Cost ≤ $1.0M):  ${(budget1M * 100).toFixed(1)}%`);
console.log(`   P(Cost ≤ $1.2M):  ${(budget1_2M * 100).toFixed(1)}%\n`);

console.log('5. Recommendation:');
const p80Duration = percentile(duration, 80);
const p80Cost = percentile(cost, 80);
console.log(`   Timeline: ${Math.ceil(p80Duration)} weeks (80% confidence)`);
console.log(`   Budget: $${Math.ceil(p80Cost / 1000)}k (80% confidence)`);

console.log('\n========================================');
console.log('       END OF SIMULATION REPORT');
console.log('========================================');

---

## Summary

### ✅ What We Covered

**MonteCarloEngine Features:**

1. **Simple Simulations**
   - Single input variable
   - Distribution sampling
   - Statistical analysis

2. **Multiple Variables**
   - Input variables (uncertain)
   - Formula variables (calculated)
   - Dependency resolution

3. **Complex Dependencies**
   - Dependency chains
   - Automatic topological sort
   - Multi-level calculations

4. **Correlations**
   - Correlation matrix definition
   - Cholesky decomposition
   - Gaussian copula
   - Preserved marginal distributions

5. **Progress Reporting**
   - Callback function
   - Real-time updates
   - Long-running simulations

6. **Real Applications**
   - Project cost/schedule risk
   - Portfolio optimization
   - Multi-variable analysis

### 🎯 Key Concepts

- **Input Variables:** Uncertain values with probability distributions
- **Formula Variables:** Calculated from other variables using JavaScript expressions
- **Dependencies:** Automatically resolved with topological sort
- **Correlations:** Model relationships between variables
- **Reproducibility:** Seeded RNG ensures repeatable results
- **Performance:** 50,000+ iterations/second typical

### 💡 Best Practices

1. **Use Seeded RNG:** `new SimpleRNG(seed)` for reproducibility
2. **Start Simple:** Single variable → multiple → formulas → correlations
3. **Validate Results:** Check mean/median match expectations
4. **Iterations:** 10,000 minimum, 50,000+ for stable percentiles
5. **Formulas:** Use JavaScript expressions (Math functions available)
6. **Correlations:** Range [-1, 1], positive-definite matrix required

### 📚 Next Steps

- **Notebook 05:** CLI Usage and Configuration
- **Build your model:** Apply to your specific use case
- **Experiment:** Try different distributions and correlations